# 01 - Contexto e modelagem

Este notebook apresenta o cenario escolhido para o trabalho, a fonte de dados utilizada e uma modelagem relacional simples para explicar as entidades `clientes`, `produtos` e `vendas`.

## Objetivos do notebook

- carregar a base `vendas.csv`;
- mostrar a estrutura dos dados;
- derivar as tabelas de negocio;
- documentar o modelo ER e o DDL proposto.


In [ ]:
import os
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LOCAL_HADOOP = PROJECT_ROOT / "windows-hadoop"
WINUTILS = LOCAL_HADOOP / "bin" / "winutils.exe"
if WINUTILS.exists():
    os.environ["HADOOP_HOME"] = str(LOCAL_HADOOP)
    os.environ["hadoop.home.dir"] = str(LOCAL_HADOOP)
else:
    print("Aviso: winutils.exe nao encontrado. Em Windows, coloque o arquivo em windows-hadoop/bin se o Spark nao iniciar.")

DATA_PATH = PROJECT_ROOT / "data" / "vendas.csv"
assert DATA_PATH.exists(), f"Arquivo nao encontrado: {DATA_PATH}"

spark = (
    SparkSession.builder
    .appName("contexto-modelagem")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_PATH))
)

raw_df.show(5, truncate=False)


## Cenario escolhido

O dominio do projeto e o de uma loja varejista. A fonte de dados possui identificadores de cliente e produto, data da venda, quantidade, preco unitario e status de pagamento.

O objetivo nao e construir um data warehouse completo, mas sim criar uma base pequena e didatica para demonstrar comandos `INSERT`, `UPDATE` e `DELETE` com `Delta Lake` e `Apache Iceberg`.


In [ ]:
vendas_df = (
    raw_df
    .withColumn("id_venda", F.col("id_venda").cast("bigint"))
    .withColumn("id_cliente", F.col("id_cliente").cast("int"))
    .withColumn("id_produto", F.col("id_produto").cast("int"))
    .withColumn("data_venda", F.to_date("data_venda"))
    .withColumn("quantidade", F.col("quantidade").cast("int"))
    .withColumn("preco_unitario", F.col("preco_unitario").cast("double"))
)

clientes_df = (
    vendas_df
    .select("id_cliente", "nome_cliente", "cidade")
    .dropDuplicates()
    .orderBy("id_cliente")
)

produtos_df = (
    vendas_df
    .select("id_produto", "nome_produto", "categoria", "preco_unitario")
    .dropDuplicates()
    .orderBy("id_produto")
)

print("Clientes")
clientes_df.show(truncate=False)

print("Produtos")
produtos_df.show(truncate=False)

print("Vendas")
vendas_df.select(
    "id_venda",
    "id_cliente",
    "id_produto",
    "data_venda",
    "quantidade",
    "status_pagamento",
).orderBy("id_venda").show(truncate=False)


## Modelo ER

![Modelo ER](../docs/assets/modelo-er.svg)

## DDL proposto

```sql
CREATE TABLE clientes (
    id_cliente INT PRIMARY KEY,
    nome_cliente STRING,
    cidade STRING
);

CREATE TABLE produtos (
    id_produto INT PRIMARY KEY,
    nome_produto STRING,
    categoria STRING,
    preco_unitario DOUBLE
);

CREATE TABLE vendas (
    id_venda BIGINT PRIMARY KEY,
    id_cliente INT,
    id_produto INT,
    data_venda DATE,
    quantidade INT,
    preco_unitario DOUBLE,
    status_pagamento STRING
);
```


In [ ]:
clientes_df.createOrReplaceTempView("clientes")
produtos_df.createOrReplaceTempView("produtos")
vendas_df.createOrReplaceTempView("vendas")

spark.sql(
    """
    SELECT
        c.nome_cliente,
        COUNT(v.id_venda) AS total_vendas,
        ROUND(SUM(v.quantidade * v.preco_unitario), 2) AS faturamento
    FROM vendas v
    JOIN clientes c ON v.id_cliente = c.id_cliente
    GROUP BY c.nome_cliente
    ORDER BY faturamento DESC
    """
).show(truncate=False)


In [ ]:
spark.stop()
